# Desafio 3 — Titanic: Machine Learning from Disaster

**Grupo:** Alpha Agents  
**Integrantes:**

- Vitor Malandrin Ferreira
- José Leonardo Alves Vilela
- Wagner Assis

## Objetivo

Construir e documentar modelos de classificação reproduzíveis para prever a
sobrevivência dos passageiros do Titanic, com pré-processamento encapsulado,
validação estratificada e geração automática das submissões.


## Responsabilidades

- **Vitor:** estrutura, pipeline, Random Forest, integração, submissão e entrega.
- **Vilela:** EDA, tratamento, engenharia de atributos, Regressão Logística e comparação.
- **Wagner:** requisitos, interpretação, documentação e validação funcional.


## 1. Importação das bibliotecas


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
STRATIFIED_CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)


## 2. Descoberta do projeto e carregamento dos dados


In [ ]:
def find_project_root() -> Path:
    # Localiza a raiz sem depender do diretório usado para abrir o notebook.
    starts = [Path.cwd().resolve()]
    if "__file__" in globals():
        starts.append(Path(__file__).resolve().parent)

    visited = set()
    for start in starts:
        for candidate in (start, *start.parents):
            if candidate in visited:
                continue
            visited.add(candidate)
            raw_dir = candidate / "data" / "raw"
            if (
                (raw_dir / "train.csv").is_file()
                and (raw_dir / "test.csv").is_file()
            ):
                return candidate

    raise FileNotFoundError(
        "Não foi possível localizar data/raw/train.csv e data/raw/test.csv "
        "a partir do diretório atual ou de seus ancestrais."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
SUBMISSION_DIR = PROJECT_ROOT / "outputs" / "submission"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print("Dados localizados em:", DATA_DIR.relative_to(PROJECT_ROOT))
print("Treino:", train_df.shape)
print("Teste:", test_df.shape)
display(train_df.head())


## 3. Validação inicial dos dados


In [ ]:
REQUIRED_TRAIN_COLUMNS = {
    "PassengerId",
    "Survived",
    "Pclass",
    "Name",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Ticket",
    "Fare",
    "Cabin",
    "Embarked",
}
REQUIRED_TEST_COLUMNS = REQUIRED_TRAIN_COLUMNS - {"Survived"}

if set(train_df.columns) != REQUIRED_TRAIN_COLUMNS:
    raise ValueError("train.csv possui colunas inesperadas.")
if set(test_df.columns) != REQUIRED_TEST_COLUMNS:
    raise ValueError("test.csv possui colunas inesperadas.")
if train_df["PassengerId"].duplicated().any():
    raise ValueError("PassengerId duplicado no treino.")
if test_df["PassengerId"].duplicated().any():
    raise ValueError("PassengerId duplicado no teste.")
if not set(train_df["Survived"].unique()).issubset({0, 1}):
    raise ValueError("Survived deve conter apenas 0 e 1.")

print("Colunas do treino:")
print(train_df.columns.tolist())

print("\nTipos:")
display(train_df.dtypes.to_frame("dtype"))

print("\nValores ausentes:")
missing = (
    train_df.isna()
    .sum()
    .to_frame("quantidade")
    .assign(percentual=lambda frame: frame["quantidade"] / len(train_df) * 100)
    .sort_values("percentual", ascending=False)
)
display(missing)

print("\nDuplicados integrais:", train_df.duplicated().sum())
display(train_df.describe(include="all").T)


## 4. Análise exploratória — Vilela

Esta seção mantém as visualizações exploratórias integradas ao notebook oficial.
As interpretações de negócio permanecem sob responsabilidade do Wagner.


In [ ]:
survival_counts = train_df["Survived"].value_counts().sort_index()
survival_counts.plot(kind="bar")
plt.title("Distribuição da sobrevivência")
plt.xlabel("Sobreviveu")
plt.ylabel("Quantidade")
plt.xticks([0, 1], ["Não", "Sim"], rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
survival_by_sex = pd.crosstab(
    train_df["Sex"],
    train_df["Survived"],
    normalize="index",
)
display(survival_by_sex)

survival_by_sex[1].plot(kind="bar")
plt.title("Taxa de sobrevivência por sexo")
plt.xlabel("Sexo")
plt.ylabel("Taxa de sobrevivência")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
survival_by_class = train_df.groupby("Pclass")["Survived"].mean()
display(survival_by_class)

survival_by_class.plot(kind="bar")
plt.title("Taxa de sobrevivência por classe")
plt.xlabel("Classe")
plt.ylabel("Taxa de sobrevivência")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Interpretação da EDA — Wagner

A interpretação dos gráficos e a redação em linguagem de negócio permanecem
reservadas ao Wagner.


## 5. Engenharia de atributos dentro do pipeline — integração Vitor


O transformador abaixo cria os atributos já usados pela equipe (`FamilySize`,
`IsAlone` e `Title`) e, opcionalmente, três atributos adicionais:

- `CabinDeck`: primeira letra da cabine, ou `Unknown` quando ausente;
- `TicketGroupSize`: frequência do bilhete aprendida somente no `fit`;
- `FarePerPerson`: tarifa dividida pelo tamanho de grupo aprendido.

A contagem de bilhetes é aprendida novamente em cada fold. Assim, validação e
teste não informam o pré-processamento do treino. Bilhetes desconhecidos recebem
tamanho 1, uma escolha conservadora e reproduzível.


In [ ]:
class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, include_additional: bool = False):
        self.include_additional = include_additional

    def fit(self, X: pd.DataFrame, y=None):
        required = {"Name", "SibSp", "Parch", "Ticket", "Fare", "Cabin"}
        missing_columns = required - set(X.columns)
        if missing_columns:
            raise ValueError(
                f"Colunas ausentes para engenharia: {sorted(missing_columns)}"
            )

        tickets = X["Ticket"].fillna("Unknown").astype(str)
        self.ticket_counts_ = tickets.value_counts().to_dict()
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not hasattr(self, "ticket_counts_"):
            raise RuntimeError("O transformador precisa ser ajustado antes do uso.")

        result = X.copy()
        result["FamilySize"] = result["SibSp"] + result["Parch"] + 1
        result["IsAlone"] = (result["FamilySize"] == 1).astype(int)

        title = (
            result["Name"]
            .str.extract(r",\s*([^.]*)\.", expand=False)
            .str.strip()
            .replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
        )
        common_titles = ["Mr", "Mrs", "Miss", "Master"]
        result["Title"] = title.where(title.isin(common_titles), "Rare")

        if self.include_additional:
            tickets = result["Ticket"].fillna("Unknown").astype(str)
            result["TicketGroupSize"] = (
                tickets.map(self.ticket_counts_).fillna(1).astype(float)
            )
            result["FarePerPerson"] = (
                result["Fare"] / result["TicketGroupSize"].clip(lower=1)
            )

            cabin = result["Cabin"].fillna("").astype(str).str.strip()
            result["CabinDeck"] = cabin.str[0].where(
                cabin.ne(""),
                "Unknown",
            )

        return result


## 6. Separação estratificada sobre os dados brutos


In [ ]:
TARGET = "Survived"
X = train_df.drop(columns=TARGET)
y = train_df[TARGET].astype(int)
X_test = test_df.copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Treino para ajuste:", X_train.shape)
print("Validação mantida fora da busca:", X_valid.shape)
print("Proporção da classe positiva:")
display(
    pd.Series(
        {
            "Treino": y_train.mean(),
            "Validação": y_valid.mean(),
        },
        name="Survived",
    ).to_frame()
)


## 7. `ColumnTransformer` e fábricas de pipelines


In [ ]:
BASE_NUMERIC_FEATURES = [
    "Age",
    "Fare",
    "SibSp",
    "Parch",
    "FamilySize",
    "IsAlone",
]
BASE_CATEGORICAL_FEATURES = [
    "Pclass",
    "Sex",
    "Embarked",
    "Title",
]
ADDITIONAL_NUMERIC_FEATURES = [
    "FarePerPerson",
    "TicketGroupSize",
]
ADDITIONAL_CATEGORICAL_FEATURES = ["CabinDeck"]


def make_preprocessor(include_additional: bool = False) -> ColumnTransformer:
    numeric_features = BASE_NUMERIC_FEATURES.copy()
    categorical_features = BASE_CATEGORICAL_FEATURES.copy()

    if include_additional:
        numeric_features += ADDITIONAL_NUMERIC_FEATURES
        categorical_features += ADDITIONAL_CATEGORICAL_FEATURES

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )


def make_model_pipeline(
    classifier,
    include_additional: bool = False,
) -> Pipeline:
    return Pipeline(
        steps=[
            (
                "features",
                TitanicFeatureEngineer(
                    include_additional=include_additional,
                ),
            ),
            (
                "preprocessor",
                make_preprocessor(
                    include_additional=include_additional,
                ),
            ),
            ("model", classifier),
        ]
    )


## 8. Modelos de referência e comparação A/B


São mantidos cinco pontos de comparação:

1. Regressão Logística, preservada como referência linear;
2. Random Forest baseline, deliberadamente pouco restringida;
3. Random Forest A, regularizada e com atributos base;
4. Random Forest B, com os mesmos parâmetros de A e engenharia adicional;
5. melhor configuração encontrada por uma grade pequena aplicada à versão B.

A comparação entre A e B usa os mesmos hiperparâmetros; portanto, a diferença
observada pode ser atribuída ao conjunto de atributos, e não ao tuning.


In [ ]:
MODEL_LOGISTIC = "Regressão Logística"
MODEL_RF_BASELINE = "RF baseline"
MODEL_RF_A = "RF A — atributos base"
MODEL_RF_B = "RF B — engenharia adicional"
MODEL_RF_GRID = "RF B — busca em grade"

logistic_model = make_model_pipeline(
    LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    )
)

rf_baseline_model = make_model_pipeline(
    RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
)


def make_regularized_rf(include_additional: bool) -> Pipeline:
    return make_model_pipeline(
        RandomForestClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        include_additional=include_additional,
    )


rf_a_model = make_regularized_rf(include_additional=False)
rf_b_model = make_regularized_rf(include_additional=True)


## 9. Busca controlada de hiperparâmetros


A grade contém somente oito combinações. Ela varia número de árvores,
profundidade e tamanho mínimo das folhas dentro de uma faixa regularizada.
O ajuste usa apenas `X_train`; o holdout `X_valid` não participa da busca.


In [ ]:
RF_PARAM_GRID = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [5, 6],
    "model__min_samples_leaf": [2, 4],
}

rf_grid_search = GridSearchCV(
    estimator=make_model_pipeline(
        RandomForestClassifier(
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        include_additional=True,
    ),
    param_grid=RF_PARAM_GRID,
    scoring="accuracy",
    cv=STRATIFIED_CV,
    n_jobs=1,
    refit=True,
    return_train_score=True,
    error_score="raise",
)
rf_grid_search.fit(X_train, y_train)

grid_results = (
    pd.DataFrame(rf_grid_search.cv_results_)
    .loc[
        :,
        [
            "param_model__n_estimators",
            "param_model__max_depth",
            "param_model__min_samples_leaf",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "rank_test_score",
        ],
    ]
    .sort_values(["rank_test_score", "std_test_score"])
    .reset_index(drop=True)
)

print("Melhores parâmetros da grade:")
print(rf_grid_search.best_params_)
display(grid_results.round(4))


## 10. Avaliação comparativa


In [ ]:
def evaluate_model(name: str, model: Pipeline) -> tuple[dict, Pipeline]:
    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)

    train_predictions = fitted_model.predict(X_train)
    valid_predictions = fitted_model.predict(X_valid)
    cv_scores = cross_val_score(
        clone(model),
        X,
        y,
        cv=STRATIFIED_CV,
        scoring="accuracy",
        n_jobs=1,
    )

    train_accuracy = accuracy_score(y_train, train_predictions)
    valid_accuracy = accuracy_score(y_valid, valid_predictions)

    metrics = {
        "Modelo": name,
        "Accuracy treino": train_accuracy,
        "Accuracy validação": valid_accuracy,
        "Precision": precision_score(
            y_valid,
            valid_predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_valid,
            valid_predictions,
            zero_division=0,
        ),
        "F1": f1_score(
            y_valid,
            valid_predictions,
            zero_division=0,
        ),
        "CV média": cv_scores.mean(),
        "CV desvio-padrão": cv_scores.std(),
        "Diferença treino-validação": train_accuracy - valid_accuracy,
    }
    return metrics, fitted_model


model_definitions = {
    MODEL_LOGISTIC: logistic_model,
    MODEL_RF_BASELINE: rf_baseline_model,
    MODEL_RF_A: rf_a_model,
    MODEL_RF_B: rf_b_model,
    MODEL_RF_GRID: rf_grid_search.best_estimator_,
}

evaluation_rows = []
fitted_models = {}

for model_name, model in model_definitions.items():
    metrics, fitted_model = evaluate_model(model_name, model)
    evaluation_rows.append(metrics)
    fitted_models[model_name] = fitted_model

results_df = (
    pd.DataFrame(evaluation_rows)
    .sort_values(
        ["CV média", "Accuracy validação", "F1"],
        ascending=False,
    )
    .reset_index(drop=True)
)
display(results_df.round(4))


## 11. Regra de seleção técnica


In [ ]:
metrics_by_model = results_df.set_index("Modelo")

rf_a_metrics = metrics_by_model.loc[MODEL_RF_A]
rf_b_metrics = metrics_by_model.loc[MODEL_RF_B]
rf_grid_metrics = metrics_by_model.loc[MODEL_RF_GRID]

additional_features_are_useful = (
    rf_b_metrics["CV média"] >= rf_a_metrics["CV média"]
    and rf_b_metrics["Accuracy validação"]
    >= rf_a_metrics["Accuracy validação"]
    and rf_b_metrics["F1"] >= rf_a_metrics["F1"]
)

rf_candidate_name = MODEL_RF_B if additional_features_are_useful else MODEL_RF_A
rf_candidate_metrics = metrics_by_model.loc[rf_candidate_name]

grid_is_better_balanced = (
    additional_features_are_useful
    and rf_grid_metrics["CV média"] > rf_candidate_metrics["CV média"] + 0.001
    and rf_grid_metrics["Accuracy validação"]
    >= rf_candidate_metrics["Accuracy validação"] - 0.01
    and rf_grid_metrics["Diferença treino-validação"]
    <= rf_candidate_metrics["Diferença treino-validação"] + 0.01
)
if grid_is_better_balanced:
    rf_candidate_name = MODEL_RF_GRID
    rf_candidate_metrics = rf_grid_metrics

final_pool = metrics_by_model.loc[
    [MODEL_LOGISTIC, rf_candidate_name]
].sort_values(
    ["CV média", "Accuracy validação", "F1"],
    ascending=False,
)
final_candidate_name = final_pool.index[0]

cv_gain_b_vs_a = rf_b_metrics["CV média"] - rf_a_metrics["CV média"]
validation_gain_b_vs_a = (
    rf_b_metrics["Accuracy validação"]
    - rf_a_metrics["Accuracy validação"]
)
f1_gain_b_vs_a = rf_b_metrics["F1"] - rf_a_metrics["F1"]

display(
    Markdown(
        f'''
**Decisão sobre os atributos adicionais:**  
A versão B variou, em relação à A, **{cv_gain_b_vs_a:+.4f}** em accuracy
média de validação cruzada, **{validation_gain_b_vs_a:+.4f}** em accuracy
do holdout e **{f1_gain_b_vs_a:+.4f}** em F1.
`CabinDeck`, `FarePerPerson` e `TicketGroupSize`
**{"foram mantidos" if additional_features_are_useful else "não foram mantidos"}**
no candidato Random Forest porque a regra exige ganho consistente nas três
medidas.

**Resultado da grade:**  
A melhor configuração da busca só substitui a referência regularizada quando
melhora a média de CV em mais de 0,001, não reduz o holdout em mais de 0,01 e
não piora materialmente o gap. Essa condição
**{"foi atendida" if grid_is_better_balanced else "não foi atendida"}**.

**Candidato Random Forest:** `{rf_candidate_name}`.  
**Candidato final para `submission.csv`:** `{final_candidate_name}`.

O critério principal é a média da validação cruzada; accuracy do holdout e F1
funcionam como desempate. A pontuação real do Kaggle não participa da escolha
porque ainda não existe.
        '''
    )
)


## 12. Sinais de overfitting


In [ ]:
overfitting_view = results_df[
    [
        "Modelo",
        "Accuracy treino",
        "Accuracy validação",
        "Diferença treino-validação",
        "CV média",
        "CV desvio-padrão",
    ]
].copy()
display(overfitting_view.round(4))

positive_gaps = overfitting_view[
    overfitting_view["Diferença treino-validação"] > 0
]
if not positive_gaps.empty:
    largest_gap_row = positive_gaps.sort_values(
        "Diferença treino-validação",
        ascending=False,
    ).iloc[0]
    print(
        "Maior sinal de overfitting:",
        largest_gap_row["Modelo"],
        f"(gap={largest_gap_row['Diferença treino-validação']:.4f})",
    )

selected_gap = metrics_by_model.loc[
    final_candidate_name,
    "Diferença treino-validação",
]
print(
    "Gap do candidato final:",
    f"{selected_gap:.4f}",
    "— valores positivos maiores pedem mais cautela.",
)


## 13. Figuras técnicas salvas


In [ ]:
comparison_metrics = [
    "Accuracy validação",
    "Precision",
    "Recall",
    "F1",
]
plot_data = results_df.set_index("Modelo")[comparison_metrics]

figure, axis = plt.subplots(figsize=(12, 6))
plot_data.plot(kind="bar", ax=axis)
axis.set_title("Comparação das métricas no conjunto de validação")
axis.set_xlabel("Modelo")
axis.set_ylabel("Métrica")
axis.set_ylim(0, 1)
axis.tick_params(axis="x", rotation=20)
axis.legend(
    loc="center left",
    bbox_to_anchor=(1.01, 0.5),
)
figure.tight_layout()

METRICS_FIGURE_PATH = FIGURES_DIR / "model_metrics_comparison.png"
figure.savefig(METRICS_FIGURE_PATH, dpi=160, bbox_inches="tight")
plt.show()
plt.close(figure)

print(
    "Figura salva:",
    METRICS_FIGURE_PATH.relative_to(PROJECT_ROOT),
)


In [ ]:
rf_candidate_fitted = fitted_models[rf_candidate_name]
rf_valid_predictions = rf_candidate_fitted.predict(X_valid)

figure, axis = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_valid,
    rf_valid_predictions,
    display_labels=["Não sobreviveu", "Sobreviveu"],
    cmap="Blues",
    colorbar=False,
    ax=axis,
)
axis.set_title(f"Matriz de confusão — {rf_candidate_name}")
axis.set_xlabel("Classe prevista")
axis.set_ylabel("Classe real")
figure.tight_layout()

CONFUSION_FIGURE_PATH = (
    FIGURES_DIR / "confusion_matrix_random_forest.png"
)
figure.savefig(CONFUSION_FIGURE_PATH, dpi=160, bbox_inches="tight")
plt.show()
plt.close(figure)

print(
    "Figura salva:",
    CONFUSION_FIGURE_PATH.relative_to(PROJECT_ROOT),
)


In [ ]:
rf_preprocessor = rf_candidate_fitted.named_steps["preprocessor"]
rf_classifier = rf_candidate_fitted.named_steps["model"]
encoded_feature_names = rf_preprocessor.get_feature_names_out()

candidate_uses_additional = rf_candidate_fitted.named_steps[
    "features"
].include_additional
original_features = (
    BASE_NUMERIC_FEATURES
    + BASE_CATEGORICAL_FEATURES
    + (
        ADDITIONAL_NUMERIC_FEATURES
        + ADDITIONAL_CATEGORICAL_FEATURES
        if candidate_uses_additional
        else []
    )
)


def recover_original_feature(encoded_name: str) -> str:
    raw_name = encoded_name.split("__", maxsplit=1)[-1]
    for feature in sorted(
        original_features,
        key=len,
        reverse=True,
    ):
        if raw_name == feature or raw_name.startswith(f"{feature}_"):
            return feature
    return raw_name


importance_df = pd.DataFrame(
    {
        "Atributo codificado": encoded_feature_names,
        "Importância": rf_classifier.feature_importances_,
    }
)
importance_df["Variável"] = importance_df[
    "Atributo codificado"
].map(recover_original_feature)
aggregated_importance = (
    importance_df.groupby("Variável", as_index=False)["Importância"]
    .sum()
    .sort_values("Importância", ascending=True)
)

figure, axis = plt.subplots(figsize=(9, 6))
axis.barh(
    aggregated_importance["Variável"],
    aggregated_importance["Importância"],
    color="#2F6B8A",
)
axis.set_title("Importância agregada das variáveis — Random Forest")
axis.set_xlabel("Importância por redução de impureza")
axis.set_ylabel("Variável")
figure.tight_layout()

IMPORTANCE_FIGURE_PATH = (
    FIGURES_DIR / "feature_importance_random_forest.png"
)
figure.savefig(IMPORTANCE_FIGURE_PATH, dpi=160, bbox_inches="tight")
plt.show()
plt.close(figure)

display(aggregated_importance.sort_values("Importância", ascending=False))
print(
    "Figura salva:",
    IMPORTANCE_FIGURE_PATH.relative_to(PROJECT_ROOT),
)


A importância por redução de impureza ajuda a inspecionar o comportamento do
Random Forest, mas não mede causalidade. A agregação reúne as categorias
produzidas pelo one-hot encoding em sua variável original.


## 14. Análise simples de erros


In [ ]:
error_feature_view = rf_candidate_fitted.named_steps[
    "features"
].transform(X_valid)
error_probabilities = rf_candidate_fitted.predict_proba(X_valid)[:, 1]

error_columns = [
    "PassengerId",
    "Pclass",
    "Sex",
    "Age",
    "Fare",
    "FamilySize",
    "Title",
]
for optional_column in [
    "CabinDeck",
    "FarePerPerson",
    "TicketGroupSize",
]:
    if optional_column in error_feature_view.columns:
        error_columns.append(optional_column)

error_analysis = error_feature_view[error_columns].copy()
error_analysis["Real"] = y_valid.to_numpy()
error_analysis["Previsto"] = rf_valid_predictions
error_analysis["Probabilidade prevista"] = error_probabilities

false_positives = error_analysis[
    (error_analysis["Real"] == 0)
    & (error_analysis["Previsto"] == 1)
].sort_values("Probabilidade prevista", ascending=False)

false_negatives = error_analysis[
    (error_analysis["Real"] == 1)
    & (error_analysis["Previsto"] == 0)
].sort_values("Probabilidade prevista", ascending=True)

print("Total de falsos positivos:", len(false_positives))
display(false_positives.head(5).round(3))

print("Total de falsos negativos:", len(false_negatives))
display(false_negatives.head(5).round(3))


**Interpretação técnica:** falsos positivos são passageiros classificados como
sobreviventes que pertenciam à classe 0; falsos negativos são sobreviventes que
o modelo não identificou. A presença dos dois tipos mostra sobreposição entre os
perfis disponíveis e limita o recall. Os exemplos acima foram deliberadamente
limitados a cinco por grupo; servem para diagnóstico do modelo, não para inferir
relações causais nem regras gerais.


## 15. Treinamento final e geração das submissões — Vitor


In [ ]:
def validate_submission(
    submission: pd.DataFrame,
    expected_ids: pd.Series,
) -> None:
    if submission.shape != (len(expected_ids), 2):
        raise ValueError(
            "A submissão deve ter 418 linhas e exatamente duas colunas."
        )
    if submission.columns.tolist() != ["PassengerId", "Survived"]:
        raise ValueError("Colunas da submissão fora do formato esperado.")
    if submission.isna().any().any():
        raise ValueError("A submissão contém valores ausentes.")
    if submission["PassengerId"].duplicated().any():
        raise ValueError("A submissão contém PassengerId duplicado.")
    if not submission["PassengerId"].reset_index(drop=True).equals(
        expected_ids.reset_index(drop=True)
    ):
        raise ValueError("PassengerId não corresponde à ordem de test.csv.")
    if not set(submission["Survived"].unique()).issubset({0, 1}):
        raise ValueError("Survived deve conter apenas 0 e 1.")


def generate_submission(model: Pipeline) -> pd.DataFrame:
    final_model = clone(model)
    if "model__n_jobs" in final_model.get_params():
        final_model.set_params(model__n_jobs=-1)
    final_model.fit(X, y)
    predictions = final_model.predict(X_test).astype(int)

    result = pd.DataFrame(
        {
            "PassengerId": test_df["PassengerId"].astype(int),
            "Survived": predictions,
        }
    )
    validate_submission(result, test_df["PassengerId"].astype(int))
    return result


rf_submission = generate_submission(
    model_definitions[rf_candidate_name]
)
final_submission = generate_submission(
    model_definitions[final_candidate_name]
)

RF_SUBMISSION_PATH = (
    SUBMISSION_DIR / "submission_random_forest.csv"
)
FINAL_SUBMISSION_PATH = SUBMISSION_DIR / "submission.csv"

rf_submission.to_csv(RF_SUBMISSION_PATH, index=False)
final_submission.to_csv(FINAL_SUBMISSION_PATH, index=False)

for path, submission_frame in [
    (RF_SUBMISSION_PATH, rf_submission),
    (FINAL_SUBMISSION_PATH, final_submission),
]:
    print("Arquivo gerado:", path.relative_to(PROJECT_ROOT))
    print("Formato:", submission_frame.shape)
    print(
        "Distribuição prevista:",
        submission_frame["Survived"].value_counts().sort_index().to_dict(),
    )

display(final_submission.head())


## 16. Resultado da submissão — Wagner

- **Pontuação:** Pendente de submissão no Kaggle.
- **Data, evidência e interpretação:** permanecem reservadas ao Wagner após a
  submissão real.


## 17. Conclusão — Wagner

A conclusão e as interpretações de negócio permanecem reservadas ao Wagner.
Os resultados técnicos reproduzíveis estão registrados nas seções anteriores.


## 18. Model Card resumida

- **Finalidade:** exercício educacional de classificação binária e comparação
  de pipelines no conjunto Titanic.
- **Dados utilizados:** `train.csv` para ajuste/avaliação e `test.csv` para
  previsão; a variável-alvo é `Survived`.
- **Métrica:** accuracy é a métrica principal, acompanhada por precision,
  recall, F1, média/desvio da validação cruzada e gap treino–validação.
- **Limitações:** amostra pequena, dados ausentes (especialmente em `Cabin`),
  classes desbalanceadas e atributos históricos incompletos.
- **Riscos de generalização:** desempenho local e validação cruzada não
  garantem resultado no Kaggle nem transferência para outras populações;
  variáveis como sexo e classe também exigem cautela ética.
- **Uso recomendado:** aprendizagem, reprodução do desafio e discussão de
  validação, leakage, overfitting e interpretabilidade.
- **Uso não recomendado:** decisões reais sobre pessoas, seguros, segurança,
  risco de vida ou qualquer contexto operacional.
- **Pontuação externa:** Pendente de submissão no Kaggle.
